# Clustering: Segmentación de Clientes Mayoristas

**Asignatura:** Ciencia de Datos (3.4.217) — UADE  
**Dataset:** Wholesale customers (UCI Machine Learning Repository)

Un distribuidor mayorista cuenta con datos del gasto anual de 440 clientes en distintas categorías de productos. El objetivo es segmentar a los clientes según su comportamiento de compra usando K-Means, y comparar los segmentos hallados con la división clásica de la empresa por `Channel` y `Region`.

## Fase I — Comprensión y Preparación de Datos

### Carga del dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)

print(df.shape)   # (440, 8)
df.head()

### Exploración rápida

In [ ]:
df.info()
df.describe().round(0)

### Selección de características

Se seleccionan únicamente las 6 variables continuas de gasto anual. `Channel` y `Region` se excluyen temporalmente: se usarán en la Fase III para el análisis cruzado.

In [ ]:
features = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
X = df[features]
X.describe().round(0)

### El reto del escalado

**Pregunta de reflexión — ¿qué impacto tendría no escalar?**

Sin escalar, la distancia euclídea queda dominada por las variables de mayor magnitud: `Fresh` alcanza valores cercanos a 112.000, mientras que `Delicassen` ronda los 1.500 en promedio. K-Means agruparía a los clientes casi exclusivamente por su gasto en frescos, y el resto de las categorías sería ruido sin peso en la distancia. Como se vio en clase, los algoritmos basados en distancia (K-Means, SVM, PCA) son sensibles a la magnitud de las variables: el escalado es esencial. Al estandarizar (media 0, desvío 1), todas las features aportan en igualdad de condiciones.

**¿Por qué StandardScaler y no MinMaxScaler?** Este dataset tiene outliers fuertes (clientes con gasto extremo). MinMaxScaler es muy sensible a outliers: un solo valor gigantesco comprime el resto de los datos en un rango diminuto y no reduce la importancia de los atípicos. StandardScaler es más robusto: los outliers siguen afectando media y desvío, pero no destruyen la escala del resto de los datos. Una alternativa aún más robusta es **RobustScaler** (resta la mediana y divide por el rango intercuartil), pensada justamente para diluir el efecto de los outliers — se compara más abajo.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=features).describe().round(2)
# Ahora todas las variables tienen media ~0 y desvío ~1

### (Opcional) Comparación con RobustScaler

Dado que el dataset tiene outliers fuertes, comparamos la calidad del clustering (coeficiente de silueta) usando StandardScaler vs RobustScaler. Si RobustScaler mejorara notablemente la silueta, convendría usarlo.

In [ ]:
from sklearn.preprocessing import RobustScaler

X_robust = RobustScaler().fit_transform(X)

for k in [2, 3, 4]:
    s_std = silhouette_score(X_scaled, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled))
    s_rob = silhouette_score(X_robust, KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_robust))
    print(f"K={k} | silueta StandardScaler: {s_std:.3f} | silueta RobustScaler: {s_rob:.3f}")

## Fase II — Modelado

### Método del Codo

In [ ]:
inertias = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, "bo-")
plt.xlabel("Número de clústeres (K)")
plt.ylabel("Inercia (WCSS)")
plt.title("Método del Codo")
plt.grid(True)
plt.show()

### Coeficiente de Silueta

In [ ]:
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouettes.append(score)
    print(f"K={k}: silueta = {score:.3f}")

plt.figure(figsize=(8, 4))
plt.plot(K_range, silhouettes, "ro-")
plt.xlabel("Número de clústeres (K)")
plt.ylabel("Coeficiente de Silueta")
plt.title("Análisis de Silueta")
plt.grid(True)
plt.show()

### Entrenamiento con el K seleccionado

In [ ]:
K_OPTIMO = 3  # ajustar según lo observado en codo + silueta

kmeans = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

df["Cluster"].value_counts().sort_index()

## Fase III — Evaluación e Interpretación

### Perfilamiento: gasto promedio real (sin escalar) por clúster

In [ ]:
perfil = df.groupby("Cluster")[features].mean().round(0)
perfil

In [ ]:
perfil_norm = perfil.div(perfil.max())  # normalizado para comparar visualmente
plt.figure(figsize=(10, 5))
sns.heatmap(perfil_norm, annot=perfil, fmt=".0f", cmap="YlGnBu")
plt.title("Gasto anual promedio por clúster")
plt.show()

### Análisis cruzado con Channel y Region

`Channel`: 1 = Horeca (Hoteles/Restaurantes/Cafés), 2 = Retail (Minoristas)  
`Region`: 1 = Lisboa, 2 = Oporto, 3 = Otra

In [ ]:
# Tabla de contingencia: Cluster vs Channel
ct_channel = pd.crosstab(df["Cluster"], df["Channel"])
ct_channel.columns = ["Horeca", "Retail"]
print("Cluster vs Channel")
print(ct_channel)
print()

# Tabla de contingencia: Cluster vs Region
ct_region = pd.crosstab(df["Cluster"], df["Region"])
ct_region.columns = ["Lisboa", "Oporto", "Otra"]
print("Cluster vs Region")
print(ct_region)

In [ ]:
# Heatmaps de las tablas de contingencia
# Color = proporción por fila (cómo se reparte cada clúster); número = conteo real de clientes
ct_channel_norm = pd.crosstab(df["Cluster"], df["Channel"], normalize="index")
ct_region_norm = pd.crosstab(df["Cluster"], df["Region"], normalize="index")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(ct_channel_norm, annot=ct_channel, fmt="d", cmap="Blues",
            cbar_kws={"label": "proporción de la fila"}, ax=axes[0])
axes[0].set_title("Cluster vs Channel")
axes[0].set_xticklabels(["Horeca", "Retail"])

sns.heatmap(ct_region_norm, annot=ct_region, fmt="d", cmap="Greens",
            cbar_kws={"label": "proporción de la fila"}, ax=axes[1])
axes[1].set_title("Cluster vs Region")
axes[1].set_xticklabels(["Lisboa", "Oporto", "Otra"])

plt.tight_layout()
plt.show()

## 4. Entregable / Reporte Final

> ⚠️ Completar con los resultados reales de las celdas anteriores antes de entregar. Los números de clúster (0, 1, 2) pueden no coincidir con el orden descripto.

### 1. ¿Cuántos clústeres elegimos y por qué?

Se eligió **K = 3**. El coeficiente de silueta alcanza su máximo en K = 2 (silueta ≈ `[completar]`), pero esa partición separa apenas un grupo pequeño de grandes compradores del resto, con poco valor comercial. El método del codo muestra un quiebre claro en K = 3 (inercia: `[completar]`), y con K = 3 la silueta (≈ `[completar]`) sigue siendo aceptable mientras los perfiles resultantes son interpretables para el negocio. Se prioriza el balance entre métrica e interpretabilidad.

### 2. Nombre comercial de cada clúster

| Clúster | Perfil de gasto | Nombre comercial |
|---|---|---|
| `[n]` | Alto en `Fresh` y `Frozen`, Channel mayoritariamente Horeca | **"Cocinas de alto volumen"** — restaurantes y hoteles que compran materia prima fresca |
| `[n]` | Alto en `Grocery`, `Milk` y `Detergents_Paper`, Channel mayoritariamente Retail | **"Supermercados de barrio"** — minoristas que revenden almacén y limpieza |
| `[n]` | Gasto bajo en todas las categorías | **"Pequeños compradores ocasionales"** — clientes chicos de bajo ticket |

### 3. Recomendación estratégica para el clúster que más gasta en congelados

El clúster con mayor gasto en `Frozen` corresponde al perfil Horeca de alto volumen. Recomendaciones para marketing:

- **Logística de cadena de frío dedicada**: entregas programadas con frecuencia fija para reducir el costo logístico y asegurar disponibilidad.
- **Descuentos por volumen en congelados**: incentivar la compra anticipada y aumentar el ticket promedio.
- **Bundles Frozen + Fresh**: estos clientes también gastan fuerte en frescos; combos cruzados aumentan la participación del distribuidor en su compra total.
- **Programa de fidelización B2B**: contratos anuales con precio preferencial a cambio de exclusividad o volumen mínimo.

### Hallazgo adicional: ¿coinciden los clústeres con la división clásica?

Las tablas de contingencia muestran que los clústeres se alinean fuertemente con `Channel` (Horeca vs. Retail) pero **no** con `Region`. Esto confirma la sospecha de la dirección: el patrón de consumo —qué compran y cuánto gastan— es el verdadero criterio de segmentación, y la ubicación geográfica no aporta información relevante para la estrategia de marketing.

### Nota metodológica: ¿por qué no hay train/test split?

El train/test split se utiliza en **aprendizaje supervisado** para evaluar si el modelo generaliza a datos no vistos y detectar overfitting. Este problema es **no supervisado**: no existe una variable objetivo que predecir ni un accuracy que medir sobre un conjunto de prueba. La validación del modelo se realiza con métricas internas de cohesión y separación de los clústeres (inercia y coeficiente de silueta).